In [2]:
import pandas as pd
import os

# Ensure output folder exists
os.makedirs("data", exist_ok=True)


In [11]:
import pandas as pd

# Load dataset
df_emotion = pd.read_csv("data/emotion_persona.csv")

# Inspect columns
print(df_emotion.columns)

# Keep only dialogue + emotion
df_emotion = df_emotion[["empathetic_dialogues", "emotion"]]

# Rename for clarity
df_emotion = df_emotion.rename(columns={
    "empathetic_dialogues": "text",
    "emotion": "label"
})

# Drop missing/duplicates
df_emotion = df_emotion.dropna(subset=["text", "label"])
df_emotion = df_emotion.drop_duplicates(subset=["text"])

# Save cleaned
df_emotion.to_csv("data/emotion_clean.csv", index=False)

print("Cleaned shape:", df_emotion.shape)
print(df_emotion.head())


Index(['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels',
       'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')
Cleaned shape: (63344, 2)
                                                text        label
0  Customer :I remember going to see the firework...  sentimental
1  Customer :This was a best friend. I miss her.\...  sentimental
2              Customer :We no longer talk.\nAgent :  sentimental
3  Customer :Was this a friend you were in love w...  sentimental
4             Customer :Where has she gone?\nAgent :  sentimental


In [ ]:
import re

def clean_text(text):
    
    text = re.sub(r"(Customer:|Agent:)", "", text)
    text = re.sub(r"(Customer\s*:|Agent\s*:)", "", text)
    text = re.sub(r"[.,]{2,}", ".", text)
    text = re.sub(r"[!?]{2,}", "!", text)
    text = re.sub(r"[,!?;:.]", r" \g<0> ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df_emotion["text"] = df_emotion["text"].astype(str).apply(clean_text)

df_emotion.to_csv("data/cleaned/emotion_clean_final.csv", index=False)

print("Sample cleaned rows:")
print(df_emotion.head(10))


Sample cleaned rows:
                                                text        label
0  I remember going to see the fireworks with my ...  sentimental
1              This was a best friend . I miss her .  sentimental
2                                We no longer talk .  sentimental
3  Was this a friend you were in love with , or j...  sentimental
4                               Where has she gone ?  sentimental
5  it feels like hitting to blank wall when i see...       afraid
6                    dont you feel so . its a wonder       afraid
7  i virtually thought so . and i used to get swe...       afraid
8                     Oh ya ? I don't really see how       afraid
9  I do actually hit blank walls a lot of times b...       afraid


Big Five Cleansing


In [2]:
import pandas as pd

# Load dataset (note: it's usually tab-separated, not comma-separated)
df = pd.read_csv("data/big_five_trait.csv", sep='\t')

# Check shape and first few columns
print("Shape:", df.shape)
print("Columns:", df.columns[:20])   # just show first 20 column names
print(df.head(3).T)   


Shape: (1015341, 110)
Columns: Index(['EXT1', 'EXT2', 'EXT3', 'EXT4', 'EXT5', 'EXT6', 'EXT7', 'EXT8', 'EXT9',
       'EXT10', 'EST1', 'EST2', 'EST3', 'EST4', 'EST5', 'EST6', 'EST7', 'EST8',
       'EST9', 'EST10'],
      dtype='object')
                             0        1        2
EXT1                       4.0      3.0      2.0
EXT2                       1.0      5.0      3.0
EXT3                       5.0      3.0      4.0
EXT4                       2.0      4.0      4.0
EXT5                       5.0      3.0      3.0
...                        ...      ...      ...
endelapse                    6       11        7
IPC                          1        1        1
country                     GB       MY       GB
lat_appx_lots_of_err   51.5448   3.1698  54.9119
long_appx_lots_of_err   0.1991  101.706  -1.3833

[110 rows x 3 columns]


In [9]:
# Step 2: Select personality items + aggregate scores
item_cols = [col for col in df.columns if col[:3] in ["EXT","EST","AGR","CSN","OPN"] and not col.endswith("_E")]
agg_cols  = [col for col in df.columns if col[:3] in ["EEX","EAG","ECO","ENE","EOP"]]  # if present

df_personality = df[item_cols + agg_cols] if agg_cols else df[item_cols]

# Drop missing values
df_personality = df_personality.dropna()

# Save a small clean sample
df_personality.head(5000).to_csv("data/cleaned/big5_clean_sample.csv", index=False)

print("✅ Cleaned personality dataset saved! Shape:", df_personality.shape)



✅ Cleaned personality dataset saved! Shape: (1013558, 50)


In [11]:
df_personality['Extraversion']    = df_personality[[f'EXT{i}' for i in range(1,11)]].mean(axis=1)
df_personality['Neuroticism']     = df_personality[[f'EST{i}' for i in range(1,11)]].mean(axis=1)
df_personality['Agreeableness']   = df_personality[[f'AGR{i}' for i in range(1,11)]].mean(axis=1)
df_personality['Conscientiousness'] = df_personality[[f'CSN{i}' for i in range(1,11)]].mean(axis=1)
df_personality['Openness']        = df_personality[[f'OPN{i}' for i in range(1,11)]].mean(axis=1)

df_traits = df_personality[['Extraversion','Neuroticism','Agreeableness','Conscientiousness','Openness']]
df_traits.head()


,Extraversion,Neuroticism,Agreeableness,Conscientiousness,Openness
0,3.0,2.4,3.1,3.2,3.3
1,3.4,2.1,3.2,3.1,2.7
2,2.9,2.6,2.8,2.8,3.1
3,2.6,2.7,3.2,2.7,3.1
4,3.5,2.3,3.0,3.2,3.6


In [12]:
#Take the mean of each trait to get a single score per trait


# Compute averages
df_big5 = pd.DataFrame()
df_big5["Extraversion"] = df[[f"EXT{i}" for i in range(1, 11)]].mean(axis=1)
df_big5["Neuroticism"] = df[[f"EST{i}" for i in range(1, 11)]].mean(axis=1)
df_big5["Agreeableness"] = df[[f"AGR{i}" for i in range(1, 11)]].mean(axis=1)
df_big5["Conscientiousness"] = df[[f"CSN{i}" for i in range(1, 11)]].mean(axis=1)
df_big5["Openness"] = df[[f"OPN{i}" for i in range(1, 11)]].mean(axis=1)

# Optional: sample smaller subset if too large
df_big5 = df_big5.sample(n=5000, random_state=42).reset_index(drop=True)

# Save cleaned version
df_big5.to_csv("data/cleaned/big5_personality_profile.csv", index=False)

print(df_big5.head())


   Extraversion  Neuroticism  Agreeableness  Conscientiousness  Openness
0           2.9          2.8            3.2                3.1       3.4
1           3.2          2.8            3.3                3.1       3.1
2           3.0          3.2            2.9                3.2       3.2
3           3.0          2.7            3.5                2.8       3.2
4           3.0          3.5            3.2                3.0       3.5
